根据标记球的位置，来测量不同大小的标记球的Z方向偏差

In [2]:
import numpy as np
import numpy.typing as npt
import pyvista as pv

In [5]:
small_pos = np.array([[228.0, 121.5, 5.3],
                      [228.4, 221.0, 6.1],
                      [303.8, 120.7, 5.0]])

big_pos = np.array([[328.8, 96.0, 9.6],
                    [204.96, 247.7, 11.2],
                    [203.8, 94.9, 10.0]])

origin_pos = (np.sum(small_pos, axis=0) + np.sum(big_pos, axis=0))/ 6

# 计算两个平面的法向量
normal_small = np.cross(small_pos[1] - small_pos[0], small_pos[2] - small_pos[0])
normal_big = np.cross(big_pos[1] - big_pos[0], big_pos[2] - big_pos[0])

# 归一化法向量
normal_small = normal_small / np.linalg.norm(normal_small)
normal_big = normal_big / np.linalg.norm(normal_big)

# 检查平行性
dot_product = np.dot(normal_small, normal_big)
if np.abs(dot_product) < 1e-6:
    print("两个平面不平行，无法计算距离")
else:
    # 计算平面方程的常数项
    d1 = -np.dot(normal_small, (small_pos[0]-origin_pos))
    d2 = -np.dot(normal_big, (big_pos[0]-origin_pos))

    # 计算两个平面之间的距离
    distance = np.abs(d2 + d1) / np.linalg.norm(normal_small)
    print(f"两个平面的距离为: {distance}")

两个平面的距离为: 4.8381952509654775


---
Foam控制中，标定Marker球在Foam表面上的重心坐标和所处的element

In [4]:
marker_pos = np.array([[137.7, -17.6, 38.8],
                       [197.1, -17.8, 39.5],
                       [213.9, -66.4, 38.6],
                       [264.4, -34.4, 38.5]], dtype=float)

corner_pos = np.array([[1.4, -153.8, 29.5],
                       [4.3, 106.8, 32.4],
                       [392.7, -156.7, 29.8],
                       [395.3, 108.7, 29.9]], dtype=float)

####################################################
# 使用最小二乘法拟合平面 ax + by + cz + d = 0, 改写为 z = ax + by + d
# 其中 a, b, c 是法向量的方向
A = np.c_[corner_pos[:, 0], corner_pos[:, 1], np.ones(corner_pos.shape[0])]
B = corner_pos[:, 2]

# 求解平面方程系数 a, b, d
a, b, d = np.linalg.lstsq(A, B, rcond=None)[0]

# 法向量可以表示为 (a, b, -1)
normal_vector = np.array([a, b, -1])
normal_vector_unit = - normal_vector / np.linalg.norm(normal_vector)

corner_project = []
for point in corner_pos:
    t = -(a * point[0] + b * point[1] - point[2] + d) / (a**2 + b**2 + 1)
    projected_point = point + t * normal_vector
    corner_project.append(projected_point)

corner_project_np = np.array(corner_project)

print(f"Plane parameters: normal unit vector: {normal_vector_unit}; \nFoam corner point in plane: \n{corner_project_np}")

####################################################
# 计算Foam在世界坐标系的齐次变换矩阵
origin_pos = np.mean(corner_project_np, axis=0)
x_axis = (corner_project_np[2] - corner_project_np[0] + corner_project_np[3] - corner_project_np[1]) / 2.
# y_axis = ((corner_project_np[1] - corner_project_np[0]) + (corner_project_np[3] - corner_project_np[2])) / 2.
z_axis = normal_vector_unit
y_axis = np.cross(z_axis, x_axis)

x_axis = x_axis / np.linalg.norm(x_axis)
y_axis = y_axis / np.linalg.norm(y_axis)
z_axis = z_axis / np.linalg.norm(z_axis)

origin_pos[0] -= 395/2
origin_pos[1] -= 270/2
origin_pos[2] -= 30      # 修正原点位置，减去Foam的厚度

R = np.column_stack((x_axis, y_axis, z_axis))
matrix_tmp = np.column_stack((x_axis, y_axis, z_axis, origin_pos))
tranfomation_matrix = np.vstack((matrix_tmp, np.array([0, 0, 0, 1])))

R_inv = np.linalg.inv(R)
t_inv = -R_inv @ origin_pos
tranfomation_matrix_inv = np.vstack((np.column_stack((R_inv, t_inv)), np.array([0, 0, 0, 1])))
print(f"Transformation matrix: \n{tranfomation_matrix}")
print(f"Transformation matrix inv: \n{tranfomation_matrix_inv}")

#####################################################
# 用于存储Marker在平面的投影点
marker_project = []

# 计算每个点到平面的投影
for point in marker_pos:
    # 计算缩放因子 t
    t = -(a * point[0] + b * point[1] - point[2] + d) / (a**2 + b**2 + 1)
    
    # 计算投影点坐标
    projected_point = point + t * normal_vector
    marker_project.append(projected_point)

# 转换为数组格式，方便查看
marker_project_np = np.array(marker_project)
print(f"Projected corner pos: \n{marker_project_np}")

######################################################
# marker在foam中的坐标 
homogeneous_marker_project = np.hstack((marker_project_np, np.ones((marker_project_np.shape[0], 1))))
homogeneous_marker_foam = (tranfomation_matrix_inv @ homogeneous_marker_project.T).T

foam_marker = homogeneous_marker_foam[:, :3] * 1.e-3
print(f"Marker pos in foam: \n{foam_marker}")
np.savetxt(f"marker_foam_pos.csv", foam_marker, fmt='%.8f', delimiter=',')

Plane parameters: normal unit vector: [ 0.00280328 -0.00568356  0.99997992]; 
Foam corner point in plane: 
[[   1.40199916 -153.80405324   30.21313691]
 [   4.29799906  106.80405684   31.68623013]
 [ 392.69803701 -156.69602009   29.09976535]
 [ 395.30196477  108.6960165    30.60086761]]
Transformation matrix: 
[[ 9.99995233e-01  1.29423357e-03  2.80328048e-03  9.25000000e-01]
 [-1.27828509e-03  9.99983031e-01 -5.68356038e-03 -1.58750000e+02]
 [-2.81058877e-03  5.67994990e-03  9.99979919e-01  4.00000000e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Transformation matrix inv: 
[[ 9.99995233e-01 -1.27828509e-03 -2.81058877e-03 -1.12679911e+00]
 [ 1.29423357e-03  9.99983031e-01  5.67994990e-03  1.58743837e+02]
 [ 2.80328048e-03 -5.68356038e-03  9.99979919e-01 -1.30485021e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Projected corner pos: 
[[137.6770281  -17.55342522  30.60551634]
 [197.07459587 -17.74849395  30.43789624]
 [213.87621243 

In [21]:
def feature_barycentric_coordinates_tet(p: npt.NDArray[np.float64], mesh_nodes: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    """
    Compute the barycentric coordinates of a point p with respect to the tetrahedron p0, p1, p2, p3.

    Parameters:
    p : ndarray of shape (3,)
        The point for which the barycentric coordinates are to be calculated.
    mesh_nodes : ndarray of shape (4, 3)
        The vertices of the tetrahedron.

    Returns:
    ndarray of shape (4,)
        The barycentric coordinates of point p with respect to the tetrahedron.
    """
    p0, p1, p2, p3 = mesh_nodes
    # Vectors relative to p0
    v0 = p1 - p0
    v1 = p2 - p0
    v2 = p3 - p0
    vp = p - p0

    # Compute the determinant of the matrix formed by v0, v1, and v2
    d00 = np.dot(v0, np.cross(v1, v2))
    if d00 == 0:
        raise ValueError("The provided points do not form a valid tetrahedron.")

    # Compute the determinants for barycentric coordinates
    d1 = np.dot(vp, np.cross(v1, v2))
    d2 = np.dot(v0, np.cross(vp, v2))
    d3 = np.dot(v0, np.cross(v1, vp))

    # Calculate barycentric coordinates
    u = 1.0 - (d1 + d2 + d3) / d00
    v = d1 / d00
    w = d2 / d00
    t = d3 / d00

    return np.array([u, v, w, t])